# 문제 2-4 : 여행 계획 분석기

여행 후기나 계획 텍스트를 입력받아 **여행지, 기간, 예산, 추천도(1-5점), 주요 활동 리스트**를 구조화된 형태로 추출한다.

**출력 파서**: `StructuredOutputParser`

**요구사항 체크**
- [x] `StructuredOutputParser` 와 `ResponseSchema` 사용
- [x] 5개의 필드 정의 — `destination`, `duration`, `budget`, `rating`, `activities`
- [x] 자연어 텍스트에서 핵심 정보 파싱 (문제 예시 후기 + 여행 계획 1개 + 다른 후기 1개)

## 1. 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정 (.env 는 프로젝트 루트에 있음)
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 전체는 출력하지 않고 설정 여부만 확인
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

## 2. LLM 설정 (ChatOpenAI)
정보 추출 작업이라 답변이 흔들리지 않도록 `temperature=0` 으로 설정한다.

In [ ]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq"(무료) 또는 "openai" — 이 값만 바꾸면 아래 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# Groq는 OpenAI 호환 API라서 ChatOpenAI에 base_url만 바꿔서 사용
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}


def get_llm(temperature: float = 0.7, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자의 ChatOpenAI 인스턴스를 생성합니다."""
    config = LLM_CONFIGS[provider or PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {(provider or PROVIDER).upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, max_retries=3)


llm = get_llm(temperature=0)
print(f"{PROVIDER} / {llm.model_name}")

## 3. 추출할 필드 5개 정의 — `ResponseSchema`
- `name` : 결과 딕셔너리의 키
- `description` : 그 키에 무엇을 넣어야 하는지 LLM 에게 주는 설명
- `type` : 형식 지시문에 표시될 값의 타입 (기본값 `string`, 활동 목록은 `List[string]`)

> LangChain 1.x 에서는 `StructuredOutputParser`, `ResponseSchema` 가 `langchain-classic` 패키지로 옮겨졌다.
> `ImportError` 가 나면 터미널에서 `uv add langchain-classic` 실행 후 커널을 재시작한다.

In [ ]:
try:
    # LangChain 1.x : StructuredOutputParser / ResponseSchema 는 langchain-classic 패키지에 있음
    from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
except ImportError:
    # LangChain 0.3.x 이하
    from langchain.output_parsers import ResponseSchema, StructuredOutputParser

response_schemas = [
    ResponseSchema(name="destination", description="여행지 (도시나 지역 이름, 예: 부산)"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="여행 예산 또는 사용한 총 경비 (예: 30만원)"),
    ResponseSchema(
        name="rating",
        description="추천도, 1~5 사이 정수 하나를 문자열로 (예: \"4\"). 점수가 직접 나오지 않으면 만족도나 기대감으로 판단",
    ),
    ResponseSchema(
        name="activities",
        description="주요 활동 리스트, '장소 + 활동' 형태의 짧은 문구 (예: [\"해운대 바다구경\", \"자갈치시장 회 먹기\"])",
        type="List[string]",
    ),
]

print(f"필드 수: {len(response_schemas)}")
for schema in response_schemas:
    print(f"- {schema.name:<12} ({schema.type})")

## 4. 출력 파서 생성 — `StructuredOutputParser`
- `from_response_schemas()` : 위의 필드 목록으로 파서를 만든다
- `get_format_instructions()` : "이 5개 키를 채운 JSON 을 마크다운 코드 블록으로 답하라"는 지시문을 만든다
- `parse()` : LLM 답변의 JSON 코드 블록을 꺼내 `dict` 로 바꾸고, 5개 키가 모두 있는지 확인한다

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

## 5. PromptTemplate 작성 & LCEL 체인 연결: `prompt | llm | output_parser`

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """당신은 여행 후기와 여행 계획을 분석하는 여행 분석가입니다.
아래 텍스트에서 여행지, 기간, 예산, 추천도(1-5점), 주요 활동을 추출하세요.
텍스트에 있는 표현을 최대한 그대로 사용하세요.

텍스트:
{travel_text}

{format_instructions}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["travel_text"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser

## 6. 실행 — 문제의 예시 입력 (부산 여행 후기)

In [ ]:
import json

example_text = (
    "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, "
    "자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. "
    "5점 만점에 4점 정도 줄 수 있을 것 같아요."
)

travel = chain.invoke({"travel_text": example_text})

print(type(travel))  # StructuredOutputParser 의 결과는 dict
print(json.dumps(travel, ensure_ascii=False, indent=4))

## 7. 다른 후기·계획 텍스트로 테스트
결과 `dict` 를 보기 좋게 정리해서 출력한다.

In [ ]:
def print_travel(info: dict) -> None:
    """추출된 여행 정보를 보기 좋게 출력"""
    rating_text = str(info["rating"]).strip()
    stars = "★" * int(rating_text) + "☆" * (5 - int(rating_text)) if rating_text in list("12345") else ""
    print(f"여행지   : {info['destination']}")
    print(f"기간     : {info['duration']}")
    print(f"예산     : {info['budget']}")
    print(f"추천도   : {rating_text}/5 {stars}")
    print("주요 활동:")
    for activity in info["activities"]:
        print(f"  - {activity}")


test_texts = [
    # 여행 계획
    "다음 달에 친구들이랑 제주도로 3박 4일 여행을 가려고 해요. 예산은 1인당 50만원 정도로 잡았고, "
    "한라산 등반하고 성산일출봉에서 일출 보고, 우도에서 자전거도 탈 계획이에요. 벌써부터 너무 기대돼요!",
    # 다른 여행 후기
    "작년 겨울에 혼자 도쿄로 4박 5일 다녀왔는데 120만원 넘게 들었어요. 시부야에서 쇼핑하고 "
    "츠키지 시장에서 초밥 먹고 디즈니랜드도 갔는데, 사람이 너무 많아서 좀 지쳤어요. 그래도 볼거리는 많아서 3점 줄게요.",
]

for i, text in enumerate([example_text, *test_texts], 1):
    info = chain.invoke({"travel_text": text})
    print(f"[입력 {i}] {text}")
    print()
    print_travel(info)
    print("-" * 70)

## 8. (선택) 직접 여행 후기·계획 입력해 보기

In [ ]:
travel_text = input("여행 후기나 계획을 입력하세요: ").strip()

if travel_text:
    info = chain.invoke({"travel_text": travel_text})
    print(json.dumps(info, ensure_ascii=False, indent=4))
    print()
    print_travel(info)
else:
    print("텍스트가 입력되지 않았습니다.")